# Exp 1 on Colab (A100 80GB)

**Runtime → Change runtime type → A100 GPU, High-RAM.** Then run the cells in order.

Where things live:
* **Drive** `MyDrive/kgcoevo/`: everything that must survive a disconnect: work dirs (`pilot/`, `main/`),
  the extracted KILT pages (`kilt/kb_pages.jsonl`) and logs (`logs/`). Every stage resumes from its JSONL output.
* **Colab local disk**: the code, KQA Pro (re-downloaded in seconds), the vLLM venv and the model weights
  (~19 GB, re-downloaded each session; reading them from Drive would be slower).

Cells 1-6 set up a session (~15-25 min the first time, mostly the KILT stream and the vLLM install).
Cell 7 runs the pilot, cell 8 the main run. Re-running a cell after a disconnect continues where it stopped.

In [ ]:
# 1. Drive and paths
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, time, urllib.request
D = '/content/drive/MyDrive/kgcoevo'           # survives disconnects
REPO = '/content/KG-agent-coevolve-project1'   # code, local disk
BRANCH = 'exp1-directed-design'
for sub in ('logs', 'kilt'):
    os.makedirs(f'{D}/{sub}', exist_ok=True)
os.environ.update(D=D, REPO=REPO, BRANCH=BRANCH)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
# 2. Code (latest commit of the branch) and the pipeline's dependencies
set -e
if [ -d "$REPO/.git" ]; then
  git -C "$REPO" fetch -q origin && git -C "$REPO" checkout -q "$BRANCH" && git -C "$REPO" reset -q --hard "origin/$BRANCH"
else
  git clone -q -b "$BRANCH" https://github.com/kyle050316/KG-agent-coevolve-project1.git "$REPO"
fi
git -C "$REPO" log --oneline -1
pip install -q -e "$REPO"

In [ ]:
%%bash
# 3. vLLM in its own venv (does not touch the notebook's packages), ~5 min
set -e
if [ ! -x /content/venv-vllm/bin/vllm ]; then
  pip install -q uv
  uv venv -q /content/venv-vllm --python 3.12
  uv pip install -q --python /content/venv-vllm/bin/python -U vllm --torch-backend=auto
fi
/content/venv-vllm/bin/python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 4. Start the vLLM server in the background (model download + load continue while cell 5 runs)
EXTRAS = ['--language-model-only --enable-prefix-caching', '--language-model-only', '--enable-prefix-caching', '']
srv = {'i': -1, 'p': None}

def up():
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5)
        return True
    except Exception:
        return False

def start_next():
    srv['i'] += 1
    extra = EXTRAS[srv['i']]
    log = open(f'{D}/logs/vllm.log', 'a')
    log.write(f'\n===== start {time.ctime()} EXTRA_ARGS={extra!r}\n'); log.flush()
    env = dict(os.environ, VLLM='/content/venv-vllm/bin/vllm', EXTRA_ARGS=extra)
    srv['p'] = subprocess.Popen(['bash', f'{REPO}/scripts/serve_vllm.sh'], stdout=log, stderr=subprocess.STDOUT, env=env)
    print('starting vLLM with', repr(extra))

if up():
    print('vLLM already running')
else:
    start_next()

In [ ]:
%%bash
# 5. Data: KQA Pro on local disk; KILT pages of KB entities on Drive (one 37 GB stream, first session only)
set -eo pipefail
mkdir -p /content/data/kqapro
HF=https://huggingface.co/datasets/drt/kqa_pro/resolve/main
for f in kb.json train.json val.json; do
  [ -s /content/data/kqapro/$f ] || curl -sSL --fail -o /content/data/kqapro/$f $HF/$f
done
ls -lh /content/data/kqapro
cd "$REPO"
if [ ! -s "$D/kilt/kb_pages.jsonl" ]; then
  # write locally first, copy when complete: a partial file on Drive would be taken as finished
  python -m experiments.exp1.prepare --config configs/main.yaml --stages kilt \
    --set paths.kqapro_dir=/content/data/kqapro paths.kilt_pages=/content/kb_pages.jsonl paths.work_dir=/content/tmp_kilt \
    2>&1 | tee "$D/logs/kilt.log"
  cp /content/kb_pages.jsonl "$D/kilt/kb_pages.jsonl"
fi
ls -lh "$D/kilt"

In [ ]:
# 6. Wait for vLLM (falls back to simpler flags if a start fails); log: MyDrive/kgcoevo/logs/vllm.log
t0 = time.time()
while not up():
    if srv['p'] is not None and srv['p'].poll() is not None:
        print('vLLM exited; see logs/vllm.log')
        if srv['i'] + 1 >= len(EXTRAS):
            raise SystemExit('vLLM could not start')
        start_next()
    if time.time() - t0 > 2400:
        raise SystemExit('vLLM not up after 40 min; see logs/vllm.log')
    time.sleep(10)
print(f'vLLM up ({time.time() - t0:.0f}s), flags {EXTRAS[srv["i"]]!r}' if srv['i'] >= 0 else 'vLLM up')
!curl -s localhost:8000/v1/models | head -c 300

In [ ]:
%%bash
# 7. Pilot (small numbers, every stage): pulls the latest code first. Report: MyDrive/kgcoevo/pilot/results/
set -eo pipefail
cd "$REPO" && git fetch -q origin && git reset -q --hard "origin/$BRANCH" && pip install -q -e . && git log --oneline -1
SET="paths.kqapro_dir=/content/data/kqapro paths.kilt_pages=$D/kilt/kb_pages.jsonl paths.work_dir=$D/pilot llm.default.concurrency=64"
CFG=configs/pilot.yaml bash scripts/run.sh prepare $SET 2>&1 | tee -a "$D/logs/pilot_prepare.log"
CFG=configs/pilot.yaml bash scripts/run.sh exp1 $SET 2>&1 | tee -a "$D/logs/pilot_exp1.log"
head -c 6000 "$D/pilot/results/exp1_report.md"

In [ ]:
%%bash
# 8. Main run. Report: MyDrive/kgcoevo/main/results/exp1_report.md
set -eo pipefail
cd "$REPO" && git fetch -q origin && git reset -q --hard "origin/$BRANCH" && pip install -q -e . && git log --oneline -1
SET="paths.kqapro_dir=/content/data/kqapro paths.kilt_pages=$D/kilt/kb_pages.jsonl paths.work_dir=$D/main llm.default.concurrency=64"
bash scripts/run.sh prepare $SET 2>&1 | tee -a "$D/logs/main_prepare.log"
bash scripts/run.sh exp1 $SET 2>&1 | tee -a "$D/logs/main_exp1.log"
head -c 6000 "$D/main/results/exp1_report.md"

In [ ]:
%%bash
# 9. Bundle results and logs into one small zip on Drive (to download and send if Drive is not synced)
cd "$D" && rm -f share.zip
zip -q -r share.zip logs pilot/results main/results pilot/exp1/usage_last_run.jsonl main/exp1/usage_last_run.jsonl \
  pilot/data/paths pilot/data/support_summary.json main/data/paths main/data/support_summary.json 2>/dev/null || true
ls -lh "$D/share.zip"